In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.metrics import f1_score

# Load Datasets

In [ ]:
batters_url = "https://github.com/willcreager/willcreager.github.io/raw/refs/heads/main/docs/data/csvs/MLBData-ABSChallengeablePitches-Batters-FullSeason.csv"
defense_url = "https://github.com/willcreager/willcreager.github.io/raw/refs/heads/main/docs/data/csvs/MLBData-ABSChallengeablePitches-Defense-FullSeason.csv"

df_batters = pd.read_csv(batters_url, encoding="latin1")
df_def = pd.read_csv(defense_url, encoding="latin1")

# Feature Engineering and One-Hot Encoding

In [ ]:
def engineer_and_encode_features(df, is_defense=False):
  df = df.dropna(subset=["plateTimeY", "plateTimeX"]).copy()

  # Spatial zone margins
  df["plateTimeY_topDiff"] = df["plateTimeY"] - df["szTop"]
  df["plateTimeY_BotDiff"] = df["plateTimeY"] - df["szBottom"]

  # Remaining challenges & net score relative to challenging side
  if is_defense:
    df["challengesLeft"] = np.where(df["inningHalf"] == 1, df["awayChallengesLeft"], df["homeChallengesLeft"])
    df["netScore"] = np.where(df["inningHalf"] == 1, df["prePlayAwayScore"] - df["prePlayHomeScore"], df["prePlayHomeScore"] - df["prePlayAwayScore"])
  else:
    df["challengesLeft"] = np.where(df["inningHalf"] == 0, df["awayChallengesLeft"], df["homeChallengesLeft"])
    df["netScore"] = np.where(df["inningHalf"] == 0, df["prePlayAwayScore"] - df["prePlayHomeScore"], df["prePlayHomeScore"] - df["prePlayAwayScore"])

  # Create categorical pitchTypeCategory label and One-Hot Encode it into numeric columns
  conditions = [
    df["pitchTypeId"].isin([5, 6, 7, 12]),
    df["pitchTypeId"].isin([2, 3, 10, 13, 14, 15]),
  ]
  df["pitchTypeCategory"] = np.select(conditions, ["Fastball", "Breaking"], default="Offspeed")

  # One-Hot Encoding step (converts categorical text column into binary numeric inputs)
  one_hot = pd.get_dummies(df["pitchTypeCategory"], prefix="pitchType", dtype=int)
  df = pd.concat([df, one_hot], axis=1)

  return df

df_batters_enc = engineer_and_encode_features(df_batters, is_defense=False)
df_def_enc = engineer_and_encode_features(df_def, is_defense=True)

# Print snapshot of One-Hot Encoding for Data Prep screenshot
print(
    df_batters_enc[["pitchTypeCategory", "pitchType_Fastball", "pitchType_Breaking", "pitchType_Offspeed"]]
    .head(5)
    .to_markdown(index=False, floatfmt=".3f")
)

# Full 16 features (including spatial coordinates) for Challenge Intent (Tasks 1 & 3)
adaboost_intent_features = [
    "inning", "prePitchBalls", "prePitchStrikes", "outs", "challengesLeft",
    "pitchType_Fastball", "pitchType_Breaking", "pitchType_Offspeed",
    "runnerOn1st", "runnerOn2nd", "runnerOn3rd",
    "plateTimeX", "plateTimeY_topDiff", "plateTimeY_BotDiff", "startSpeed", "netScore"
]

# 13 situational/velocity features (excluding spatial coordinates to prevent leakage) for Challenge Outcome (Tasks 2 & 4)
adaboost_outcome_features = [
    "inning", "prePitchBalls", "prePitchStrikes", "outs", "challengesLeft",
    "pitchType_Fastball", "pitchType_Breaking", "pitchType_Offspeed",
    "runnerOn1st", "runnerOn2nd", "runnerOn3rd",
    "startSpeed", "netScore"
]

# Create function to run AdaBoost pipeline

In [ ]:
def evaluate_adaboost_cv(X, y, task_label, use_balancing=False, max_depth=1):
  print(f"\n==================================================================")
  print(f"🚀 CV ADABOOST PIPELINE: {task_label.upper()}")
  print(f"==================================================================")

  # Strictly disjoint 80/20 stratified train/test split (Final test set stays untouched during tuning)
  X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
  )
  print(f"Disjoint Splits Created: Train = {X_train.shape} | Untouched Holdout Test = {X_test.shape}")

  # Hyperparameter grid for rounds (n_estimators) and learning_rate
  n_estimators_list = [10, 25, 50, 100, 150]
  max_est = max(n_estimators_list)
  eval_indices = {n - 1: n for n in n_estimators_list}
  learning_rates = [0.1, 0.5, 1.0]

  # Use 3-fold CV on massive intent datasets (>50k rows) for fast runtime, 5-fold on smaller outcome sets
  n_splits = 3 if len(X_train) > 50000 else 5
  cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
  scoring_metric = "f1_macro" if use_balancing else "accuracy"

  # Store scores: {(lr, n_est): [fold1_score, fold2_score, ...]}
  fold_score_map = {(lr, n_est): [] for lr in learning_rates for n_est in n_estimators_list}

  print(f"Tuning n_estimators & learning_rate via {n_splits}-Fold Stratified CV ({scoring_metric})...")

  for lr in learning_rates:
    for train_idx, val_idx in cv.split(X_train, y_train):
      X_tr, y_tr = X_train.iloc[train_idx], y_train.iloc[train_idx]
      X_va, y_va = X_train.iloc[val_idx], y_train.iloc[val_idx]

      stump = DecisionTreeClassifier(max_depth=max_depth, random_state=42)
      ada = AdaBoostClassifier(
        estimator=stump,
        n_estimators=max_est,
        learning_rate=lr,
        random_state=42
      )

      if use_balancing:
        sw = compute_sample_weight(class_weight="balanced", y=y_tr)
        ada.fit(X_tr, y_tr, sample_weight=sw)
      else:
        ada.fit(X_tr, y_tr)

      # Evaluate all n_estimators milestones in a single pass using staged_predict
      for step_idx, preds_va in enumerate(ada.staged_predict(X_va)):
        if step_idx in eval_indices:
          n_est = eval_indices[step_idx]
          if use_balancing:
            score = f1_score(y_va, preds_va, average="macro")
          else:
            score = accuracy_score(y_va, preds_va)
          fold_score_map[(lr, n_est)].append(score)

  tuning_records = []
  best_score = -1.0
  best_params = None

  for lr in learning_rates:
    for n_est in n_estimators_list:
      scores = fold_score_map[(lr, n_est)]
      mean_cv_score = np.mean(scores)
      std_cv_score = np.std(scores)
      tuning_records.append({
        "learning_rate": lr,
        "n_estimators": n_est,
        "mean_cv_score": mean_cv_score,
        "std_cv_score": std_cv_score
      })
      if mean_cv_score > best_score:
        best_score = mean_cv_score
        best_params = {"learning_rate": lr, "n_estimators": n_est}

  tuning_df = pd.DataFrame(tuning_records)
  print(f"\n🏆 Best Hyperparameters from {n_splits}-Fold CV: n_estimators = {best_params['n_estimators']}, "
        f"learning_rate = {best_params['learning_rate']} (CV {scoring_metric} = {best_score:.4f})")

  # ---- PLOT 1: VALIDATION CURVE (CV Score vs. Boosting Rounds across Learning Rates) ----
  plt.figure(figsize=(8.5, 5))
  for lr in learning_rates:
    subset = tuning_df[tuning_df["learning_rate"] == lr]
    plt.plot(subset["n_estimators"], subset["mean_cv_score"], marker="o", linewidth=2.2,
              label=f"Learning Rate = {lr}")
    plt.fill_between(subset["n_estimators"],
                      subset["mean_cv_score"] - subset["std_cv_score"],
                      subset["mean_cv_score"] + subset["std_cv_score"], alpha=0.12)

  plt.title(f"AdaBoost Validation Curve: {task_label}\n({n_splits}-Fold CV {scoring_metric} vs. Number of Rounds)", fontweight="bold")
  plt.xlabel("Number of Boosting Rounds (n_estimators)")
  plt.ylabel(f"Mean CV {scoring_metric}")
  plt.xticks(n_estimators_list)
  plt.legend()
  plt.grid(True, linestyle=":", alpha=0.6)
  plt.tight_layout()
  plt.show()

  # ---- FINAL EVALUATION: Fit winning model on full X_train, evaluate ONCE on untouched X_test ----
  final_stump = DecisionTreeClassifier(max_depth=max_depth, random_state=42)
  final_ada = AdaBoostClassifier(
    estimator=final_stump,
    n_estimators=best_params["n_estimators"],
    learning_rate=best_params["learning_rate"],
    random_state=42
  )

  if use_balancing:
    full_sw = compute_sample_weight(class_weight="balanced", y=y_train)
    final_ada.fit(X_train, y_train, sample_weight=full_sw)
  else:
    final_ada.fit(X_train, y_train)

  y_test_pred = final_ada.predict(X_test)
  test_acc = accuracy_score(y_test, y_test_pred)
  cm = confusion_matrix(y_test, y_test_pred)

  print(f"\n✅ FINAL UNTOUCHED HOLDOUT TEST EVALUATION:")
  print(f"   • Holdout Test Accuracy: {test_acc:.4f}")
  print("   • Confusion Matrix:")
  print(cm)
  print("\n   • Classification Report:")
  print(classification_report(y_test, y_test_pred))

  imp_df = pd.DataFrame({
    "Feature": X.columns,
    "Importance": final_ada.feature_importances_
  }).sort_values(by="Importance", ascending=False)
  print("🔍 Top 5 Most Important Features in Final AdaBoost Ensemble:")
  print(imp_df.head(5).to_string(index=False))

  # ---- PLOT 2: CONFUSION MATRIX HEATMAP ----
  plt.figure(figsize=(6, 5))
  sns.heatmap(cm, annot=True, fmt="d", cmap="Oranges",
              xticklabels=final_ada.classes_, yticklabels=final_ada.classes_)
  plt.title(f"AdaBoost Holdout Confusion Matrix: {task_label}\n(Test Acc = {test_acc:.4f})", fontweight="bold")
  plt.xlabel("Predicted Label")
  plt.ylabel("Actual Label")
  plt.tight_layout()
  plt.show()

  return final_ada, tuning_df, imp_df

# Run AdaBoost pipeline to predict batter challenge intent

In [ ]:
# Task 1. Batter Challenge Intent (16 features, Balanced)
# Takes about 3 minutes to run
df_b_clean = df_batters_enc.dropna(subset=adaboost_intent_features + ["hasABSChallenge"]).copy()
X_b_intent = df_b_clean[adaboost_intent_features]
y_b_intent = df_b_clean["hasABSChallenge"].map({1: "Challenged", 0: "Not Challenged"})
evaluate_adaboost_cv(X_b_intent, y_b_intent, "Batter Challenge Intent (Balanced)", use_balancing=True)

# Run AdaBoost pipeline to predict batter challenge outcome

In [ ]:
# Task 2. Batter Challenge Outcome (13 features — no spatial zone coordinates)
df_b_chal = df_b_clean[df_b_clean["hasABSChallenge"] == 1].dropna(subset=["isOverturned"])
X_b_out = df_b_chal[adaboost_outcome_features]
y_b_out = df_b_chal["isOverturned"].map({1: "Overturned", 0: "Confirmed"})
evaluate_adaboost_cv(X_b_out, y_b_out, "Batter Challenge Outcome", use_balancing=False)

# Run AdaBoost pipeline to predict defense challenge intent

In [ ]:
# Task 3. Defense Challenge Intent (16 features, Balanced)
# Takes about 9 minutes to run
df_d_clean = df_def_enc.dropna(subset=adaboost_intent_features + ["hasABSChallenge"]).copy()
X_d_intent = df_d_clean[adaboost_intent_features]
y_d_intent = df_d_clean["hasABSChallenge"].map({1: "Challenged", 0: "Not Challenged"})
evaluate_adaboost_cv(X_d_intent, y_d_intent, "Defense Challenge Intent (Balanced)", use_balancing=True)

# Run AdaBoost pipeline to predict defense challenge outcome

In [ ]:
# 4. Defense Challenge Outcome (13 features — no spatial zone coordinates)
df_d_chal = df_d_clean[df_d_clean["hasABSChallenge"] == 1].dropna(subset=["isOverturned"])
X_d_out = df_d_chal[adaboost_outcome_features]
y_d_out = df_d_chal["isOverturned"].map({1: "Overturned", 0: "Confirmed"})
evaluate_adaboost_cv(X_d_out, y_d_out, "Defense Challenge Outcome", use_balancing=False)

# Create diagrams for website

In [ ]:
# ==============================================================================
# FIGURE 1: BAGGING (PARALLEL) VS. ADABOOST SEQUENTIAL REWEIGHTING
# ==============================================================================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Left Panel: Bagging (Parallel)
ax1.set_xlim(0, 10)
ax1.set_ylim(0, 10)
ax1.axis("off")
ax1.set_title("1A: Bagging (Parallel Independent Trees)", fontsize=12, fontweight="bold", pad=12)

box_data = dict(boxstyle="round,pad=0.45", fc="#e8f4f8", ec="#1f77b4", lw=1.5)
box_tree = dict(boxstyle="round,pad=0.45", fc="#d4edda", ec="#2ca02c", lw=1.5)
box_vote = dict(boxstyle="round,pad=0.5", fc="#fff3cd", ec="#d39e00", lw=1.8)

ax1.text(5, 8.8, "Original Training Dataset\n(Equal Weights)", ha="center", va="center", fontsize=10, fontweight="bold", bbox=box_data)
for idx, x_pos in enumerate([2.0, 5.0, 8.0], start=1):
  ax1.text(x_pos, 6.2, f"Bootstrap\nSample {idx}", ha="center", va="center", fontsize=9.5, bbox=box_data)
  ax1.text(x_pos, 3.6, f"Independent\nTree {idx}", ha="center", va="center", fontsize=9.5, fontweight="bold", bbox=box_tree)
  ax1.annotate("", xy=(x_pos, 6.95), xytext=(5, 8.05), arrowprops=dict(arrowstyle="->", lw=1.5, color="black"))
  ax1.annotate("", xy=(x_pos, 4.35), xytext=(x_pos, 5.45), arrowprops=dict(arrowstyle="->", lw=1.5, color="black"))
  ax1.annotate("", xy=(5, 1.85), xytext=(x_pos, 2.85), arrowprops=dict(arrowstyle="->", lw=1.5, color="black"))

ax1.text(5, 1.1, "Unweighted Majority Vote\n(Reduces Variance)", ha="center", va="center", fontsize=10, fontweight="bold", bbox=box_vote)

# Right Panel: AdaBoost (Sequential Error Reweighting)
ax2.set_xlim(0, 10)
ax2.set_ylim(0, 10)
ax2.axis("off")
ax2.set_title("1B: AdaBoost (Sequential Error Reweighting)", fontsize=12, fontweight="bold", pad=12)

box_stump = dict(boxstyle="round,pad=0.45", fc="#fce8e6", ec="#d93025", lw=1.5)
box_final = dict(boxstyle="round,pad=0.5", fc="#e6f4ea", ec="#137333", lw=1.8)

ax2.text(2.2, 8.5, "Round 1: Equal Weights\n(w = 1/N for all pitches)", ha="center", va="center", fontsize=9.5, bbox=box_data)
ax2.text(7.5, 8.5, "Stump 1 (h₁)\nWeight α₁", ha="center", va="center", fontsize=9.5, fontweight="bold", bbox=box_stump)
ax2.annotate("", xy=(5.7, 8.5), xytext=(4.3, 8.5), arrowprops=dict(arrowstyle="->", lw=1.8, color="black"))

ax2.text(2.2, 5.7, "Round 2: Boost Misclassified\n(Hard pitches get higher weight)", ha="center", va="center", fontsize=9.5, bbox=box_data)
ax2.text(7.5, 5.7, "Stump 2 (h₂)\nWeight α₂", ha="center", va="center", fontsize=9.5, fontweight="bold", bbox=box_stump)
ax2.annotate("Increase error\nweights", xy=(2.2, 6.5), xytext=(6.2, 7.4),
             arrowprops=dict(arrowstyle="->", lw=1.6, color="crimson", linestyle="--"),
             fontsize=8.5, fontweight="bold", color="crimson", ha="center")
ax2.annotate("", xy=(5.7, 5.7), xytext=(4.5, 5.7), arrowprops=dict(arrowstyle="->", lw=1.8, color="black"))

ax2.text(2.2, 2.9, "Round T: Focus on Toughest\nBorderline Edge Cases", ha="center", va="center", fontsize=9.5, bbox=box_data)
ax2.text(7.5, 2.9, "Stump T (h_T)\nWeight α_T", ha="center", va="center", fontsize=9.5, fontweight="bold", bbox=box_stump)
ax2.annotate("Reweight\nerrors", xy=(2.2, 3.7), xytext=(6.2, 4.6),
             arrowprops=dict(arrowstyle="->", lw=1.6, color="crimson", linestyle="--"),
             fontsize=8.5, fontweight="bold", color="crimson", ha="center")
ax2.annotate("", xy=(5.7, 2.9), xytext=(4.4, 2.9), arrowprops=dict(arrowstyle="->", lw=1.8, color="black"))

ax2.text(5.0, 0.9, r"Final Ensemble: $H(x) = \mathrm{sign}\left(\sum_{t=1}^{T} \alpha_t h_t(x)\right)$",
         ha="center", va="center", fontsize=10, fontweight="bold", bbox=box_final)

plt.subplots_adjust(left=0.04, right=0.96, bottom=0.06, top=0.88, wspace=0.15)
plt.suptitle("Figure 1: Parallel Bagging vs. Sequential Boosting (AdaBoost) Mechanics", fontsize=14, fontweight="bold", y=0.97)
plt.savefig("adaboost_sequential_reweighting.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# ==============================================================================
# FIGURE 2: WEAK DECISION STUMPS COMBINING INTO STRONG ENSEMBLE BOUNDARY
# ==============================================================================
fig, ax = plt.subplots(figsize=(10, 5.5))
ax.set_xlim(0, 12)
ax.set_ylim(0, 10)
ax.axis("off")

stump_box = dict(boxstyle="round,pad=0.5", fc="#fff8e1", ec="#f57c00", lw=1.6)
alpha_box = dict(boxstyle="round,pad=0.35", fc="#e3f2fd", ec="#1565c0", lw=1.5)
out_box = dict(boxstyle="round,pad=0.6", fc="#e8f5e9", ec="#2e7d32", lw=2.0)

stumps = [
  (2.2, 7.8, "Stump 1 (max_depth=1)\nplateTimeX <= 0.78 ft", r"$\alpha_1 = 0.85$" + "\n(Low Error)"),
  (2.2, 5.0, "Stump 2 (max_depth=1)\nplateTimeY_BotDiff <= -0.04 ft", r"$\alpha_2 = 0.62$" + "\n(Moderate Error)"),
  (2.2, 2.2, "Stump T (max_depth=1)\nprePitchStrikes >= 2", r"$\alpha_T = 0.41$" + "\n(Higher Error)")
]

for y_pos, title, alpha_txt in [(s[1], s[2], s[3]) for s in stumps]:
  ax.text(2.3, y_pos, title, ha="center", va="center", fontsize=10, fontweight="bold", bbox=stump_box)
  ax.text(6.0, y_pos, alpha_txt, ha="center", va="center", fontsize=9.5, fontweight="bold", bbox=alpha_box)
  ax.annotate("", xy=(4.8, y_pos), xytext=(4.0, y_pos), arrowprops=dict(arrowstyle="->", lw=1.8, color="black"))
  ax.annotate("", xy=(8.3, 5.0), xytext=(7.2, y_pos), arrowprops=dict(arrowstyle="->", lw=1.8, color="black"))

ax.text(10.1, 5.0, "Weighted Majority Vote\n\nStrong Classifier H(x)\nPredicts: Challenged vs.\nNot Challenged",
        ha="center", va="center", fontsize=10.5, fontweight="bold", bbox=out_box)

ax.set_title("Figure 2: Combining Weak Decision Stumps via Error-Weighted Voting (α)", fontsize=13, fontweight="bold", pad=12)
plt.tight_layout()
plt.savefig("adaboost_stumps_weighted_vote.png", dpi=300, bbox_inches="tight")
plt.show()